# 03 - Training the LSTM Sequence Detector

This notebook walks through training the TrajectoryLSTM model that powers MLShield's Layer 2 detection. The LSTM learns to detect anomalous event sequences in ML infrastructure trajectories.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix
from pathlib import Path

from mlshield.detectors.models.lstm_detector import TrajectoryLSTM, EventFeaturizer

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Data Preparation

Load the benchmark dataset and featurize all trajectories into (50, 32) feature matrices.

In [ ]:
with open('../benchmark/data/mlshield_benchmark_v1.json') as f:
    dataset = json.load(f)

featurizer = EventFeaturizer()
X, y = [], []

for traj in dataset:
    label = 0 if traj['label'] == 'benign' else 1
    features = featurizer.featurize_trajectory(traj['events'], max_len=50)
    X.append(features)
    y.append(label)

X = np.array(X)
y = np.array(y)

print(f'Dataset shape: {X.shape}')
print(f'Benign: {(y == 0).sum()}, Malicious: {(y == 1).sum()}')
print(f'Feature vector: {X.shape[2]}-dim per event, {X.shape[1]} events per trajectory')

## 2. Train/Val/Test Split

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.176, random_state=42, stratify=y_trainval
)

print(f'Train: {len(X_train)} ({(y_train == 0).sum()} benign, {(y_train == 1).sum()} malicious)')
print(f'Val:   {len(X_val)} ({(y_val == 0).sum()} benign, {(y_val == 1).sum()} malicious)')
print(f'Test:  {len(X_test)} ({(y_test == 0).sum()} benign, {(y_test == 1).sum()} malicious)')

## 3. PyTorch Dataset & DataLoader

In [ ]:
class TrajectoryDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_ds = TrajectoryDataset(X_train, y_train)
val_ds = TrajectoryDataset(X_val, y_val)
test_ds = TrajectoryDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 4. Model Architecture

The TrajectoryLSTM uses:
- 2-layer LSTM with 64 hidden units
- Attention mechanism for weighted sequence aggregation
- Classifier head with dropout

In [ ]:
model = TrajectoryLSTM(input_dim=32, hidden_dim=64, num_layers=2, dropout=0.2)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable: {trainable:,}')
print()
print(model)

## 5. Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

epochs = 30
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_auc = 0.0
best_state = None

for epoch in range(epochs):
    # Train
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        pred = model(batch_X)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate
    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            pred = model(batch_X)
            val_loss += criterion(pred, batch_y).item()
            val_preds.extend(pred.squeeze().tolist())
            val_labels.extend(batch_y.squeeze().tolist())
    val_loss /= len(val_loader)

    if isinstance(val_preds, float):
        val_preds = [val_preds]
    auc = roc_auc_score(val_labels, val_preds)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(auc)

    if auc > best_auc:
        best_auc = auc
        best_state = model.state_dict().copy()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | AUC: {auc:.4f}')

print(f'\nBest validation AUC: {best_auc:.4f}')
model.load_state_dict(best_state)

## 6. Training History

In [ ]:
import pandas as pd

hist_df = pd.DataFrame(history)
hist_df.index += 1
hist_df.index.name = 'epoch'
print(hist_df.to_string())

## 7. Test Set Evaluation

In [ ]:
model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        pred = model(batch_X)
        test_preds.extend(pred.squeeze().tolist())
        test_labels.extend(batch_y.squeeze().tolist())

test_preds = np.array(test_preds)
test_labels = np.array(test_labels)

auc = roc_auc_score(test_labels, test_preds)
binary_preds = (test_preds > 0.5).astype(int)
prec, rec, f1, _ = precision_recall_fscore_support(test_labels, binary_preds, average='binary', zero_division=0)
cm = confusion_matrix(test_labels, binary_preds)

print('=== Test Set Results ===')
print(f'AUC:       {auc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1 Score:  {f1:.4f}')
print(f'\nConfusion Matrix:')
print(f'  TN={cm[0][0]}, FP={cm[0][1]}')
print(f'  FN={cm[1][0]}, TP={cm[1][1]}')

## 8. Save Model

In [ ]:
output_path = '../benchmark/data/models/lstm_detector.pt'
torch.save(model.state_dict(), output_path)
print(f'Model saved to: {output_path}')
print(f'Model size: {Path(output_path).stat().st_size / 1024:.1f} KB')

## Summary

1. **TrajectoryLSTM** learns temporal patterns from 32-dimensional event feature sequences
2. **Attention mechanism** allows the model to focus on the most suspicious events in a trajectory
3. The model achieves strong AUC on the benchmark, complementing Layer 1 static rules
4. Combined with Isolation Forest (0.7/0.3 weighting), it forms MLShield's Layer 2 detector